# Converting <span style="font-variant:small-caps;">Html</span> to Text

This notebook shows how we can use the standard Python `re` module
to extract the text that is embedded in an <span style="font-variant:small-caps;">Html</span> file.  
In order to be concise, it only supports a small subset of 
<span style="font-variant:small-caps;">Html</span>.  Below is the content of my old
<a href="http://wwwlehre.dhbw-stuttgart.de/~stroetma/">web page</a> that I had used when I was still working at the DHBW Stuttgart.  The goal of this notebook is to write 
a sequence of regular expressions to extract the text from this web page.

In [ ]:
data = \
'''
<!doctype html>
<html>
  <head>
    <meta charset="utf-8">
    <title>Homepage of Prof. Dr. Karl Stroetmann</title>
    <link type="text/css" rel="stylesheet" href="style.css" />
    <link href="http://fonts.googleapis.com/css?family=Rochester&subset=latin,latin-ext"
          rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Pacifico&subset=latin,latin-ext"
          rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Cabin+Sketch&subset=latin,latin-ext" rel="stylesheet" type="text/css">
    <link href="http://fonts.googleapis.com/css?family=Sacramento" rel="stylesheet" type="text/css">
  </head>
  <body>
    <hr/>

    <div id="table">
      <header>
        <h1 id="name">Prof. Dr. Karl Stroetmann</h1>
      </header>

      <div id="row1">
        <div class="right">
          <a id="dhbw" href="http://www.ba-stuttgart.de">Duale Hochschule Baden-W&uuml;rttemberg</a>
          <br/>Coblitzallee 1-9
          <br/>68163 Mannheim
          <br/>Germany
	  <br>
          <br/>Office: &nbsp;&nbsp;&nbsp; Raum 344B
          <br/>Phone:&nbsp;&nbsp;&nbsp; +49 621 4105-1376
          <br/>Fax:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; +49 621 4105-1194
          <br/>Skype: &nbsp;&nbsp;&nbsp; karlstroetmann
        </div>  


        <div id="links">
          <strong class="some">Some links:</strong>
          <ul class="inlink">
            <li class="inlink">
	      My <a class="inlink" href="https://github.com/karlstroetmann?tab=repositories">lecture notes</a>,
              as well as the programs presented in class, can be found
              at <br>
              <a class="inlink" href="https://github.com/karlstroetmann?tab=repositories">https://github.com/karlstroetmann</a>.
              
            </li>
            <li class="inlink">Most of my papers can be found at <a class="inlink" href="https://www.researchgate.net/">researchgate.net</a>.</li>
            <li class="inlink">The programming language SetlX can be downloaded at <br>
              <a href="http://randoom.org/Software/SetlX"><tt class="inlink">http://randoom.org/Software/SetlX</tt></a>.
            </li>
          </ul>
        </div>
      </div>
    </div>
    
    <div id="intro">
      As I am getting old and wise, I have to accept the limits of
      my own capabilities.  I have condensed these deep philosophical
      insights into a most beautiful pearl of poetry.  I would like 
      to share these humble words of wisdom:
      
      <div class="poetry">
        I am a teacher by profession,    <br>
        mostly really by obsession;      <br>
        But even though I boldly try,    <br>
        I just cannot teach <a href="flying-pig.jpg" id="fp">pigs</a> to fly.</br>
        Instead, I slaughter them and fry.
      </div>
      
      <div class="citation">
        <div class="quote">
          Any sufficiently advanced poetry is indistinguishable from divine wisdom.
        </div>
        <div id="sign">His holiness Pope Hugo &#8555;.</div>
      </div>
    </div>
</div>

</body>
</html>
'''

In [ ]:
from IPython.core.display import HTML

In [ ]:
HTML(data)

## Imports

We will use the standard `re` module to remove the 
<span style="font-variant:small-caps;">Html</span> tags and extract the text that
is embedded in the <span style="font-variant:small-caps;">Html</span> shown above.
In order to support named <span style="font-variant:small-caps;">Html5</span> entities 
we also need to import the dictionary `html5` from the module `html.entities`.

In [ ]:
import re
from html.entities import html5

## Text Extraction Pipeline

We will approach this problem by applying a sequence of regular expression substitutions (`re.sub`). This functional approach passes the text through a pipeline where each step systematically removes unwanted elements or decodes entities.  We begin by copying our initial <span style="font-variant:small-caps;">Html</span> string.

In [ ]:
text = data

### 1. Removing `<head>` and `<script>` Elements

Once we are inside an <span style="font-variant:small-caps;">Html</span> header or inside of some
*JavaScript* code, the rules change because we want to completely discard their contents. We use the regular expression `<head>.*?</head>` to match the header block and `<script>.*?</script>` to match any embedded scripts. 

The flag `re.DOTALL` ensures that the `.` character matches newlines as well, allowing us to remove multi-line blocks. The `re.IGNORECASE` flag handles both lowercase and uppercase tag names.

In [ ]:
text = re.sub(r'<head>.*?</head>'    , '', text, flags=re.IGNORECASE | re.DOTALL)
text = re.sub(r'<script>.*?</script>', '', text, flags=re.IGNORECASE | re.DOTALL)

In [ ]:
print(text)

### 2. Removing Remaining <span style="font-variant:small-caps;">Html</span> Tags

With the `<head>` and `<script>` blocks out of the way, we can safely remove all remaining <span style="font-variant:small-caps;">Html</span> tags. The regular expression `<[^>]+>` matches any string that starts with the character `<` and ends with the character `>`. Between these two characters, there has to be a nonzero number of characters that are different from the character `>`. These matched tags are simply replaced by an empty string.

In [ ]:
text = re.sub(r'<[^>]+>', '', text)
print(text)

### 3. Resolving Named Entities

The regular expression `&[a-zA-Z]+;?` searches for <span style="font-variant:small-caps;">Html</span> entity names. These are strings that start with the character `&` followed by the name of the entity, optionally followed by the character `;`. For example, `&auml;` is the entity name that specifies the German umlaut `ä`.

We define a replacement function `replace_named` that extracts the entity name and uses the `html5` dictionary to return the corresponding Unicode character. Notice that we use modern Python type annotations (`re.Match`) to declare the parameter type clearly without requiring obsolete `typing` module imports.

In [ ]:
def replace_named(match: re.Match) -> str:
    s = match.group(0)
    if s[-1] == ';':
        entity_name = s[1:-1]   # chop off '&' at the start and ';' at the end
    else:
        entity_name = s[1:]     # only chop '&' off
    
    return html5.get(entity_name, s)

text = re.sub(r'&[a-zA-Z]+;?', replace_named, text)

In [ ]:
print(text)

### 4. Resolving Numeric Unicode Entities

The regular expression `&#[0-9]+;?` searches for <span style="font-variant:small-caps;">Html</span> entities that specify a unicode character numerically. The corresponding strings start with the character `&` followed by the character `#` followed by digits and are optionally ended by the character `;`.

Our replacement function `replace_unicode` isolates the numeric part of the match and converts it using the built-in `chr` function. For example, `chr(128034)` returns the character `'🐢'`.

In [ ]:
def replace_unicode(match: re.Match) -> str:
    s = match.group(0)
    if s[-1] == ';':
        number_str = s[2:-1]    # chop off '&#' at the start and ';' at the end
    else:
        number_str = s[2:]      # chop off '&#' at the start
        
    try:
        return chr(int(number_str))
    except ValueError:
        return s

text = re.sub(r'&#[0-9]+;?', replace_unicode, text)

### 5. Cleaning Up Linebreaks

Finally, the regular expression `(\s*\n\s*)+` matches groups of whitespace containing at least one newline character. We condense these groups into a single newline character to clean up the formatting of our extracted text output.

In [ ]:
text = re.sub(r'(\s*\n\s*)+', '\n', text)
print(text.strip())